_Célula 1_

# Gráficos de energia eólica por tipo de região
Leitura de `eolica_consolidado.csv` e geração de gráficos de barras por `tipo_region` (bioma, estado e país).

In [1]:
# Célula 2
import pandas as pd
import matplotlib.pyplot as plt
import kaleido
import numpy as np
import plotly.graph_objects as go
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter
import textwrap

salvar_grafico = True

df = pd.read_csv('eolica_consolidado.csv')
df.head()

,year,version,area_ha,tipo_region,nome_region,sigla_region
0,2016,0-4-12-spt-4,4.671005,bioma,Mata Atlântica,MAT
1,2017,0-4-12-spt-4,43.568322,bioma,Mata Atlântica,MAT
2,2018,0-4-12-spt-4,87.540817,bioma,Mata Atlântica,MAT
3,2019,0-4-12-spt-4,226.438065,bioma,Mata Atlântica,MAT
4,2020,0-4-12-spt-4,258.019166,bioma,Mata Atlântica,MAT


In [2]:
# Célula 3
# Paleta sequencial usada em todos os gráficos (clara = valor menor/ano mais
# antigo, escura = valor maior/ano mais recente) — tons de cinza definidos
# pelo usuário (#403d3e, #393637, #333031, #2c2a2b), completados com 3 tons
# mais claros seguindo a mesma progressão.
# CMAP_CINZA = mcolors.LinearSegmentedColormap.from_list(
#     'eolica_cinza', ['#555253', '#4e4b4c', '#474445', '#403d3e', '#393637', '#333031', '#2c2a2b']
# )
MAP_LARANJA = mcolors.LinearSegmentedColormap.from_list(
    'eolica_laranja', ['#F7A173', '#F3732F', '#F05D0E', '#CD500C', "#B3470D", "#953A08", "#7F3006"]
)

def rampa_cores(n: int, inverso: bool = False):
    """`n` cores igualmente espaçadas na paleta sequencial (clara → escura)."""
    posicoes = np.linspace(0.05, 0.95, n)
    if inverso:
        posicoes = posicoes[::-1]
    return [MAP_LARANJA(p) for p in posicoes]


def formata_ptbr(valor, casas: int = 0) -> str:
    """Formata número no padrão brasileiro (ponto como separador de milhar)."""
    s = f'{valor:,.{casas}f}'
    return s.replace(',', '_').replace('.', ',').replace('_', '.')


def quebra_nome(nome: str, largura: int = 12) -> str:
    """Quebra nomes de região longos em 2 linhas para caber no eixo X."""
    return '\n'.join(textwrap.wrap(nome, width=largura, break_long_words=False))


FORMATADOR_PTBR = FuncFormatter(lambda v, _: formata_ptbr(v))

print('Utilitários de estilo definidos.')

Utilitários de estilo definidos.


In [3]:
# Célula 4
# Requer o pacote `kaleido` instalado (pip install -U kaleido) para exportar
# as figuras Plotly como imagem.
try:
    from google.colab import files as colab_files
    EM_COLAB = True
except ImportError:
    EM_COLAB = False


def salvar_grafico_alta_resolucao(fig, nome_arquivo, dpi=500, formato='png', baixar=True):
    """Salva um gráfico Plotly (`fig`) em alta resolução (300 ou 450 dpi) e,
    se `baixar=True` e estiver rodando no Colab, dispara o download do
    arquivo.

    O Plotly/kaleido não trabalha com DPI diretamente — a resolução da
    imagem exportada é controlada por `scale` (multiplicador do
    width/height definidos no layout da figura). Aqui `scale` é calculado
    como dpi/96, usando 96 dpi como referência padrão de tela.
    """
    escala = dpi / 96
    caminho = nome_arquivo if nome_arquivo.lower().endswith(f'.{formato}') else f"{nome_arquivo}.{formato}"
    fig.write_image(caminho, scale=escala)
    print(f"Gráfico salvo em '{caminho}' (dpi≈{dpi}, scale={escala:.2f})")

    if baixar and EM_COLAB:
        colab_files.download(caminho)


# uso: salvar_grafico_alta_resolucao(fig, 'evolucao_area_por_bioma_eolica', dpi=300)
# ou, para mais nitidez: dpi=450

_Célula 5_

## Resumo
Painel de biomas — evolução da área por bioma ao longo dos anos, com destaque de valor e participação percentual nos anos de 2021 e 2025.

In [4]:
# Célula 6
df_bioma = df[df['tipo_region'] == 'bioma']
pivot_bioma = df_bioma.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
totais_anuais = pivot_bioma.sum(axis=1)

# Limite fixo do eixo Y — None mantém o comportamento antigo (calculado a
# partir do maior total anual, com folga de 45%); passe uma tupla/lista
# (min, max) para fixar a escala manualmente.
YLIM = (0, 32000)

cores_map = {'Caatinga': "#AB420A", 'Cerrado': "#BE4E12", 'Mata Atlântica': "#F27836", 'Pampa': "#F7A173"}
# Caatinga domina o total (~85%), deixando os outros 3 biomas como fatias tão
# finas na barra empilhada que nenhum texto cabe legível dentro delas — por
# isso viram um bloco de texto à parte, acima da barra (ver textos_totais).
BIOMAS_MINORITARIOS = ['Cerrado', 'Mata Atlântica', 'Pampa']
# BIOMAS_MINORITARIOS = []

fig = go.Figure()

for bioma in pivot_bioma.columns:
    textos = []
    for ano, v in zip(pivot_bioma.index, pivot_bioma[bioma]):
        # Caatinga mostra o rótulo (valor + %) desde o início da série, não
        # só em 2021/2025 — é o único bioma fora de BIOMAS_MINORITARIOS.
        if bioma not in BIOMAS_MINORITARIOS and v > 0:
            valor_k = f"{v/1000:.1f} K".replace('.', ',')
            pct = (v / totais_anuais.loc[ano]) * 100
            textos.append(f"{valor_k}<br>{pct:.0f}%")
        else:
            textos.append("")

    fig.add_trace(go.Bar(
        x=pivot_bioma.index, y=pivot_bioma[bioma], name=bioma,
        marker_color=cores_map.get(bioma, '#ccc'),
        text=textos, textposition='inside', insidetextanchor='middle',
        textfont=dict(color='white', size=16),
        hovertemplate='<b>Bioma:</b> ' + bioma + '<br><b>Ano:</b> %{x}<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
    ))

# Ancora o bloco de texto um pouco acima do topo real da barra (1,08x), para
# a última linha do bloco não encostar/sobrepor a própria barra.
y_texto = totais_anuais * 1.08

textos_totais = []
for ano, total in zip(totais_anuais.index, totais_anuais):
    total_k = f"{total/1000:.1f} K".replace('.', ',')
    # 2025 mantém o detalhamento por bioma minoritário (como já estava); os
    # demais anos passam a exibir também o total, só que sem o detalhamento.
    # Todas as linhas do rótulo de topo saem em preto (sem cor por bioma).
    if ano == 2026:
        linhas = [f"<b>{total_k}</b>"]
        for bioma in BIOMAS_MINORITARIOS:
            v = pivot_bioma.loc[ano, bioma]
            if v <= 0:
                continue
            valor_k = f"{v/1000:.1f} K".replace('.', ',')
            pct = (v / total) * 100
            linhas.append(f'{bioma} {valor_k} ({pct:.0f}%)')
        textos_totais.append("<br>".join(linhas))
    else:
        textos_totais.append(f"<b>{total_k}</b>")

fig.add_trace(go.Scatter(
    x=totais_anuais.index, y=y_texto, mode='text', text=textos_totais,
    textposition='top center', textfont=dict(size=14, color='black'),
    showlegend=False, hoverinfo='skip',
    # Sem isso, o bloco de texto do último ano (2025, encostado na borda
    # direita do gráfico) é cortado pela área de plotagem.
    cliponaxis=False,
))

fig.update_layout(barmode='stack', 
                  title_text="Evolução da Área de Energia Eólica por Bioma", 
                  yaxis_title= "hectares",
                  template="plotly_white", 
                  width=900, height=650,
    legend=dict(orientation="h", y=-0.05, x=0.5, xanchor="center", font=dict(size=16), entrywidth=0.24, entrywidthmode='fraction'),
    margin=dict(t=100, b=150, l=60, r=150), xaxis=dict(tickmode='linear'),
    yaxis=dict(range=list(YLIM) if YLIM is not None else [0, totais_anuais.max() * 1.45]))
fig.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig, 'evolucao_area_por_bioma_eolica', dpi=600)

Salvando o grafico com DPI 450
Gráfico salvo em 'evolucao_area_por_bioma_eolica.png' (dpi≈600, scale=6.25)


In [5]:
# Célula 6A
# Variante em gráfico de área empilhada (em vez de barras) do mesmo painel de biomas.
df_bioma_area = df[df['tipo_region'] == 'bioma']
pivot_bioma_area = df_bioma_area.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
totais_anuais_area = pivot_bioma_area.sum(axis=1)

cores_map_area = {'Caatinga': "#AB420A", 'Cerrado': "#BE4E12", 'Mata Atlântica': "#F27836", 'Pampa': "#F7A173"}
BIOMAS_MINORITARIOS_AREA = ['Cerrado', 'Mata Atlântica', 'Pampa']

fig_area = go.Figure()

for bioma in pivot_bioma_area.columns:
    # Rótulo da Caatinga (valor em K nos extremos 2016/2025) fica oculto —
    # só o bloco de total por ano (abaixo) continua visível.
    textos = ["" for _ in pivot_bioma_area.index]

    fig_area.add_trace(go.Scatter(
        x=pivot_bioma_area.index, y=pivot_bioma_area[bioma], name=bioma,
        mode='lines+text', stackgroup='biomas',
        line=dict(width=0.5, color=cores_map_area.get(bioma, '#ccc')),
        fillcolor=cores_map_area.get(bioma, '#ccc'),
        text=textos, textposition='bottom center',
        textfont=dict(color='white', size=16),
        hovertemplate='<b>Bioma:</b> ' + bioma + '<br><b>Ano:</b> %{x}<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
    ))

# Bloco de texto com o total do ano, ancorado acima da área empilhada — apenas
# o total em negrito para todos os anos (rótulos de Cerrado, Mata Atlântica e
# Pampa ficam ocultos também em 2025).
y_texto_area = totais_anuais_area * 1.08

textos_totais_area = [f"<b>{total/1000:.1f} K</b>".replace('.', ',') for total in totais_anuais_area]

fig_area.add_trace(go.Scatter(
    x=totais_anuais_area.index, y=y_texto_area, mode='text', text=textos_totais_area,
    textposition='top center', textfont=dict(size=14, color='black'),
    showlegend=False, hoverinfo='skip',
    cliponaxis=False,
))

fig_area.update_layout(
    barmode='stack', title_text="Evolução da Área de Energia Eólica por Bioma (Área)", template="plotly_white",
    width=650, height=650,
    legend=dict(orientation="h", y=-0.05, x=0.5, xanchor="center", font=dict(size=16), entrywidth=0.24, entrywidthmode='fraction'),
    margin=dict(t=100, b=150, l=60, r=150), xaxis=dict(tickmode='linear'),
    yaxis=dict(range=[0, totais_anuais_area.max() * 1.45])
)
fig_area.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig_area, 'evolucao_area_por_bioma_eolica_area', dpi=450)

Salvando o grafico com DPI 450


Gráfico salvo em 'evolucao_area_por_bioma_eolica_area.png' (dpi≈450, scale=4.69)


In [ ]:
# Célula 7
ANO_INICIAL, ANO_FINAL = 2016, 2025
lst_Nodeste = ['Rio Grande do Norte', 'Bahia', 'Piauí', 'Ceará', 'Paraíba','Pernambuco', 'Sergipe', 'Maranhão']

def calcula_crescimento(df, tipo_region, ano_inicial=ANO_INICIAL, ano_final=ANO_FINAL):
    """Área e % de crescimento entre `ano_inicial` e `ano_final` para cada
    região do `tipo_region` informado ('bioma', 'estado' ou 'pais')."""
    sub = df[df['tipo_region'] == tipo_region]
    pivot = sub.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)

    area_inicial = pivot.loc[ano_inicial]
    area_final = pivot.loc[ano_final]

    tabela = pd.DataFrame({
        'nome_region': area_inicial.index,
        f'area_{ano_inicial}': area_inicial.values,
        f'area_{ano_final}': area_final.values,
    })
    # np.where evita divisão por zero nas regiões que ainda não tinham área em ano_inicial
    tabela['crescimento_pct'] = np.where(
        tabela[f'area_{ano_inicial}'] > 0,
        (tabela[f'area_{ano_final}'] / tabela[f'area_{ano_inicial}']) * 100,
        np.nan,
    )
    return tabela.sort_values('crescimento_pct', ascending=False, na_position='last').reset_index(drop=True)


def formata_tabela_crescimento(tabela):
    tabela_fmt = tabela.copy()
    for col in tabela_fmt.columns:
        if col.startswith('area_'):
            tabela_fmt[col] = tabela_fmt[col].apply(formata_ptbr)
    tabela_fmt['crescimento_pct'] = tabela_fmt['crescimento_pct'].apply(
        lambda p: f"{p:.0f}%".replace('.', ',') if pd.notna(p) else "—"
    )
    return tabela_fmt


# Crescimento nacional (país)
crescimento_pais = calcula_crescimento(df, 'pais')
pct_brasil = crescimento_pais['crescimento_pct'].iloc[0]
print(f"Crescimento da área de energia eólica no Brasil entre {ANO_INICIAL} e {ANO_FINAL}: "
      + f"{pct_brasil:.0f}%".replace('.', ','))
display(formata_tabela_crescimento(crescimento_pais))

# Crescimento por estado
print(f"\nCrescimento por estado entre {ANO_INICIAL} e {ANO_FINAL}:")
display(formata_tabela_crescimento(calcula_crescimento(df, 'estado')))

# Crescimento por bioma
print(f"\nCrescimento por bioma entre {ANO_INICIAL} e {ANO_FINAL}:")
display(formata_tabela_crescimento(calcula_crescimento(df, 'bioma')))

# Nordeste agregado — soma das áreas de lst_Nodeste (tratados como uma única
# região), mesmo espírito do ESTADOS_GRUPO da célula 14, mas somando os 8
# estados nordestinos entre ANO_INICIAL e ANO_FINAL.
pivot_estado_nordeste = df[df['tipo_region'] == 'estado'].pivot_table(
    index='year', columns='nome_region', values='area_ha', aggfunc='sum'
).fillna(0)

area_nordeste_inicial = pivot_estado_nordeste.loc[ANO_INICIAL, lst_Nodeste].sum()
area_nordeste_final = pivot_estado_nordeste.loc[ANO_FINAL, lst_Nodeste].sum()
crescimento_nordeste_pct = (
    (area_nordeste_final / area_nordeste_inicial - 1) * 100
    if area_nordeste_inicial > 0 else np.nan
)

tabela_nordeste = pd.DataFrame({
    'nome_region': ['Nordeste (' + ', '.join(lst_Nodeste) + ')'],
    f'area_{ANO_INICIAL}': [area_nordeste_inicial],
    f'area_{ANO_FINAL}': [area_nordeste_final],
    'crescimento_pct': [crescimento_nordeste_pct],
})

print(f"\nNordeste agregado ({len(lst_Nodeste)} estados) entre {ANO_INICIAL} e {ANO_FINAL}:")
display(formata_tabela_crescimento(tabela_nordeste))

Crescimento da área de energia eólica no Brasil entre 2016 e 2025: 500%


,nome_region,area_2016,area_2025,crescimento_pct
0,Brasil,4.783,28.679,500%



Crescimento por estado entre 2016 e 2025:


,nome_region,area_2016,area_2025,crescimento_pct
0,Paraíba,3,1.241,41032%
1,Rio Grande do Sul,99,1.703,1613%
2,Rio Grande do Norte,1.038,9.194,785%
3,Ceará,406,2.765,580%
4,Piauí,755,3.443,356%
5,Bahia,1.941,8.489,337%
6,Pernambuco,532,1.265,138%
7,Maranhão,0,451,—
8,Rio de Janeiro,0,15,—
9,Santa Catarina,0,45,—



Crescimento por bioma entre 2016 e 2025:


,nome_region,area_2016,area_2025,crescimento_pct
0,Mata Atlântica,5,358,7560%
1,Pampa,95,1.696,1692%
2,Caatinga,3.605,24.445,578%
3,Cerrado,1.078,2.180,102%



Nordeste agregado (8 estados) entre 2016 e 2025:


,nome_region,area_2016,area_2025,crescimento_pct
0,"Nordeste (Rio Grande do Norte, Bahia, Piauí, C...",4.676,26.891,475%


In [41]:
areaRN= 5280900
areaBA = 56473300
areaRN_eolica = 9194
areaBA_eolica = 8489

print(f"proporsão eolica na bahia {10000 * (areaBA_eolica/areaBA)}")
print(f"proporsão eolica na Rio Grande do Norte {1000 * (areaRN_eolica/areaRN)}")




proporsão eolica na bahia 1.5031882323150942
proporsão eolica na Rio Grande do Norte 1.7409911189380598


In [7]:
area_total = 28679
areaRN_eolica = 9194
areaBA_eolica = 8489
print("porcentagem eolica RN ", 100*(areaRN_eolica/ area_total))
print("porcentagem eolica Bahia ", 100*(areaBA_eolica/ area_total))
print("porcentagem eolica RN com respeito a BA ", round(100*(1 -  areaBA_eolica/ areaRN_eolica), 2), " %")

porcentagem eolica RN  32.058300498622685
porcentagem eolica Bahia  29.600055789950837
porcentagem eolica RN com respeito a BA  7.67  %


In [16]:
# Célula 8
import plotly.graph_objects as go

# 1. Preparação dos dados para o último ano disponível
df_bioma = df[df['tipo_region'] == 'bioma']
pivot_bioma = df_bioma.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano = int(pivot_bioma.index.max())
serie_ultimo_ano = pivot_bioma.loc[ultimo_ano].sort_values(ascending=False)

cores_map = {'Caatinga': "#AB420A", 'Cerrado': "#BE4E12", 'Mata Atlântica': "#F27836", 'Pampa': "#F7A173"}

# 2. Criar Gráfico de Pizza
# rotation=207: como Caatinga domina (~85%), as 3 fatias pequenas (Cerrado,
# Pampa, Mata Atlântica) ficam todas espremidas perto de um mesmo ângulo — com
# rotation=30 elas caem no topo do gráfico e o rótulo colide com o título.
# Girando para 207 essa aglomeração vai para a parte de baixo, longe do
# título; automargin+pull dão espaço extra para o Plotly não sobrepor texto.
fig = go.Figure(data=[go.Pie(
    labels=serie_ultimo_ano.index,
    values=serie_ultimo_ano.values,
    marker=dict(colors=[cores_map.get(b) for b in serie_ultimo_ano.index]),
    textinfo='percent+label+value',
    rotation=107,
    texttemplate='%{label}<br>%{percent:.0%}<br>%{value:,.2f} ha',
    insidetextfont=dict(size=17),
    outsidetextfont=dict(size=17),
    automargin=True,
    pull=[0, 0.03, 0.05, 0.08],
    hovertemplate='<b>Bioma:</b> %{label}<br><b>Área:</b> %{value:,.2f} ha<extra></extra>'
)])

# 3. Layout
fig.update_layout(
    title_text=f"Participação por Bioma na Área de Energia Eólica ({ultimo_ano})",
    title_font=dict(size=24),
    template="plotly_white",
    width=900,
    height=600,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.15,
        xanchor="center",
        x=0.5,
        entrywidth=0.3,
        entrywidthmode='fraction',
        font=dict(size=16)
    ),
    margin=dict(t=100, b=100, l=50, r=50)
)

fig.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig, 'proporcao_area_por_bioma_eolica', dpi=450)

Salvando o grafico com DPI 450
Gráfico salvo em 'proporcao_area_por_bioma_eolica.png' (dpi≈450, scale=4.69)


In [17]:
# Célula 8A
# Anéis concêntricos (não fatias de uma mesma pizza): cada bioma ocupa uma
# faixa de raio própria — Caatinga (mais externo) > Cerrado > Pampa > Mata
# Atlântica (mais interno) — e todos os arcos partem do mesmo ângulo inicial
# (0°/12h), com o comprimento proporcional à participação do bioma no total.
# Por isso os arcos ficam com tamanhos bem diferentes entre si (esperado).
import plotly.graph_objects as go

df_bioma = df[df['tipo_region'] == 'bioma']
pivot_bioma = df_bioma.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano = int(pivot_bioma.index.max())
serie_ultimo_ano = pivot_bioma.loc[ultimo_ano].sort_values(ascending=False)
total_ano = serie_ultimo_ano.sum()

# cores_map = {
#     'Caatinga': '#1b1a1a',
#     'Cerrado': '#3C3C3C',
#     'Mata Atlântica': '#7A7878',
#     'Pampa': '#C1C0C0'
# }

# Do mais externo (Caatinga) para o mais interno (Mata Atlântica)
ORDEM_ANEIS = ['Caatinga', 'Cerrado', 'Pampa', 'Mata Atlântica']
N = len(ORDEM_ANEIS)
ESPACO = 0.02  # respiro entre um anel e o próximo
ESPESSURA = (1 - (N - 1) * ESPACO) / N  # espessura de cada anel, para o conjunto ocupar de r=0 até r=1

fig_aneis = go.Figure()
for i, bioma in enumerate(ORDEM_ANEIS):
    valor = serie_ultimo_ano.get(bioma, 0)
    frac = valor / total_ano
    arco_graus = frac * 360
    base_raio = 1 - i * (ESPESSURA + ESPACO) - ESPESSURA

    fig_aneis.add_trace(go.Barpolar(
        r=[ESPESSURA],
        theta=[arco_graus / 2],  # todos os arcos começam em 0°; o centro do arco é a metade do seu próprio comprimento
        width=[arco_graus],
        base=[base_raio],
        name=bioma,
        marker=dict(color=cores_map.get(bioma, '#ccc'), line=dict(color='white', width=1)),
        hovertemplate=f'<b>Bioma:</b> {bioma}<br><b>Área:</b> {valor:,.2f} ha<br><b>Participação:</b> {frac*100:.1f}%<extra></extra>',
    ))

# --- rótulos: lista empilhada à esquerda do anel, texto grande e em negrito
# (tamanho decrescente conforme o %), igual ao layout de referência ---
FONTE_ROTULO = {'Caatinga': 30, 'Cerrado': 23, 'Pampa': 19, 'Mata Atlântica': 17}

anotacoes = []
Y_INICIAL, PASSO_Y = 0.88, 0.13
for i, bioma in enumerate(ORDEM_ANEIS):
    valor = serie_ultimo_ano.get(bioma, 0)
    frac = valor / total_ano
    anotacoes.append(dict(
        x=0.04, y=Y_INICIAL - i * PASSO_Y, xref='paper', yref='paper',
        xanchor='left', align='left',
        text=f"<b>{bioma} {frac*100:.0f}%</b>",
        showarrow=False,
        font=dict(size=FONTE_ROTULO.get(bioma, 18), color='#1b1a1a'),
    ))

fig_aneis.update_layout(
    title_text=f"Participação por Bioma na Área de Energia Eólica ({ultimo_ano})",
    title_font=dict(size=24),
    template="plotly_white",
    width=900, height=800,
    showlegend=False,
    annotations=anotacoes,
    polar=dict(
        # anel deslocado para a direita, deixando espaço à esquerda para os rótulos
        domain=dict(x=[0.38, 0.98], y=[0.05, 0.95]),
        radialaxis=dict(visible=False, range=[0, 1]),
        angularaxis=dict(visible=False, rotation=90, direction='clockwise'),
    ),
    margin=dict(t=90, b=30, l=30, r=30),
)

fig_aneis.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig_aneis, 'proporcao_area_por_bioma_eolica_aneis_concentricos', dpi=450)

Salvando o grafico com DPI 450
Gráfico salvo em 'proporcao_area_por_bioma_eolica_aneis_concentricos.png' (dpi≈450, scale=4.69)


_Célula 9_

## Estados
Gráfico de barras verticais — um grupo por estado (ordenado do maior para o menor pela área do ano mais recente) e uma barra por ano dentro do grupo, com rótulo nos anos de 2021 e 2025 — mesmo estilo do gráfico de estados de UFV.

In [44]:
# Célula 10
import plotly.graph_objects as go

df_estado = df[df['tipo_region'] == 'estado']
pivot_estado = df_estado.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano_estado = pivot_estado.index.max()
ordem_desc = pivot_estado.loc[ultimo_ano_estado].sort_values(ascending=False).index
pivot_estado = pivot_estado[ordem_desc]

estados = pivot_estado.columns.tolist()
# Mostra só até Maranhão — Santa Catarina, Sergipe e Rio de Janeiro têm área
# residual (poucas dezenas de ha) e não agregam à leitura do gráfico.
idx_corte = estados.index('Maranhão') + 1
estados = estados[:idx_corte]
anos = pivot_estado.index.tolist()

# Paleta própria deste gráfico: os primeiros anos precisam ficar nitidamente
# mais claros que os últimos, então em vez da CMAP_CINZA padrão (toda em
# tons escuros/médios) usa-se aqui uma rampa que começa em cinza bem claro
# e termina nos mesmos tons escuros da paleta original.
CMAP_LARANJA_ESTADOS = mcolors.LinearSegmentedColormap.from_list(
    'eolica_laranja', ['#F7A173', '#F3732F', '#F05D0E', '#CD500C', "#B3470D", "#953A08", "#7F3006"]
)
cores_anos_plotly = [
    f'rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {a})'
    for r, g, b, a in [CMAP_LARANJA_ESTADOS(p) for p in np.linspace(0.05, 0.95, len(anos))]
]

# Maior valor do grupo, para dar espaço suficiente acima da barra mais alta
# (senão o Plotly encolhe automaticamente o rótulo dela para caber)
max_valor_estado = max(pivot_estado.loc[ano, e] for ano in anos for e in estados)

fig = go.Figure()

for i, ano in enumerate(anos):
    # Regra: Destaque apenas em 2021 e 2025 com fonte 26
    font_size = 14 if ano in [2025] else 1 # 2021, 
    exibir_texto = ano in [2025]   # 2021, 

    valores = [pivot_estado.loc[ano, e] for e in estados]
    textos = [formata_ptbr(v) if (v > 0 and exibir_texto) else "" for v in valores]

    fig.add_trace(
        go.Bar(
            x=estados,
            y=valores,
            name=str(ano),
            marker_color=cores_anos_plotly[i],
            # text=textos,
            # textposition='outside',
            # textfont=dict(size=font_size, color='black'),
            # textangle=-90,
            # Sem isso, o Plotly encolhe o texto das barras mais altas (perto
            # do topo do gráfico) para caber no espaço disponível, deixando o
            # rótulo minúsculo/ilegível mesmo com font_size fixo em 26.
            constraintext='none',
            hovertemplate='<b>Estado:</b> %{x}<br><b>Ano:</b> ' + str(ano) + '<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
        )
    )

fig.update_layout(
    title="Estados - Evolução da Área de Energia Eólica (Destaque 2021/2025)",
    yaxis_title= 'hectares',
    width=950, height=650, 
    template="plotly_white", barmode='group',
    legend=dict(
        orientation="h", y=-0.05, x=0.5, xanchor="center",
        font=dict(size=16), entrywidth=0.19, entrywidthmode='fraction'
    ),
    margin=dict(t=80, b=150, l=60, r=40),
    # Espaço extra acima da barra mais alta para o rótulo vertical não ser cortado
    yaxis=dict(range=[0, max_valor_estado * 1.10])
)
fig.update_xaxes(tickangle=0)
fig.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig, 'evolucao_anoXarea_por_estado_eolica', dpi=600)

Salvando o grafico com DPI 450
Gráfico salvo em 'evolucao_anoXarea_por_estado_eolica.png' (dpi≈600, scale=6.25)


_Célula 11_

### Participação percentual por estado (ano mais recente)
Percentual da área de energia eólica de cada estado em relação ao total somado de todos os estados.

In [28]:
# Célula 12
df_estado_pct = df[df['tipo_region'] == 'estado']
pivot_estado_pct = df_estado_pct.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano_pct = pivot_estado_pct.index.max()

serie_estado_pct = pivot_estado_pct.loc[ultimo_ano_pct].sort_values(ascending=False)
total_estados_pct = serie_estado_pct.sum()

tabela_pct_estado = pd.DataFrame({
    'estado': serie_estado_pct.index,
    'area_ha': serie_estado_pct.values,
})
tabela_pct_estado['percentual'] = tabela_pct_estado['area_ha'] / total_estados_pct * 100

# versão formatada (pt-BR) só para exibição — tabela_pct_estado continua numérica
tabela_pct_estado_fmt = tabela_pct_estado.copy()
tabela_pct_estado_fmt['area_ha'] = tabela_pct_estado_fmt['area_ha'].apply(formata_ptbr)
tabela_pct_estado_fmt['percentual'] = tabela_pct_estado_fmt['percentual'].apply(lambda p: f"{p:.1f}%".replace('.', ','))

display(tabela_pct_estado_fmt)

,estado,area_ha,percentual
0,Rio Grande do Norte,9.194,"32,1%"
1,Bahia,8.489,"29,6%"
2,Piauí,3.443,"12,0%"
3,Ceará,2.765,"9,6%"
4,Rio Grande do Sul,1.703,"5,9%"
5,Pernambuco,1.265,"4,4%"
6,Paraíba,1.241,"4,3%"
7,Maranhão,451,"1,6%"
8,Santa Catarina,45,"0,2%"
9,Sergipe,43,"0,1%"


_Célula 13_

### Estados agrupados — Top 4 (ano mais recente)
Soma da área e percentual conjunto dos 4 principais estados (Rio Grande do Norte, Bahia, Piauí e Ceará) em relação ao total geral.

In [29]:
# Célula 14
ESTADOS_GRUPO = ['Rio Grande do Norte', 'Bahia']  # , 'Piauí', 'Ceará'

df_estado_grupo = df[df['tipo_region'] == 'estado']
pivot_estado_grupo = df_estado_grupo.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano_grupo = pivot_estado_grupo.index.max()

serie_estado_grupo = pivot_estado_grupo.loc[ultimo_ano_grupo]
total_geral_estados = serie_estado_grupo.sum()
area_grupo = serie_estado_grupo[ESTADOS_GRUPO].sum()
percentual_grupo = area_grupo / total_geral_estados * 100

tabela_grupo_estados = pd.DataFrame({
    'estados': [' + '.join(ESTADOS_GRUPO)],
    'area_ha': [area_grupo],
    'percentual': [percentual_grupo],
})

# versão formatada (pt-BR) só para exibição
tabela_grupo_estados_fmt = tabela_grupo_estados.copy()
tabela_grupo_estados_fmt['area_ha'] = tabela_grupo_estados_fmt['area_ha'].apply(formata_ptbr)
tabela_grupo_estados_fmt['percentual'] = tabela_grupo_estados_fmt['percentual'].apply(lambda p: f"{p:.1f}%".replace('.', ','))

display(tabela_grupo_estados_fmt)

,estados,area_ha,percentual
0,Rio Grande do Norte + Bahia,17.684,"61,7%"


_Célula 15_

## País (Brasil)
Como há apenas uma região do tipo `pais` (Brasil), o gráfico mostra a área por ano.

In [30]:
# Célula 16
import plotly.graph_objects as go

# 1. Preparação dos dados
df_pais = df[df['tipo_region'] == 'pais']
pivot_pais = df_pais.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum')
serie_brasil = pivot_pais['Brasil']

anos = serie_brasil.index.tolist()
valores = serie_brasil.values.tolist()

# Cores baseadas na rampa definida anteriormente (claro -> escuro)
cores_br = [f'rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {a})' for r, g, b, a in rampa_cores(len(anos))]

# 2. Criar Gráfico de Linha Interativo
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=anos,
    y=valores,
    mode='lines+markers+text',
    line=dict(color='#d9702f', width=4),
    marker=dict(
        color=cores_br,
        size=12,
        line=dict(color='white', width=1.5)
    ),
    text=[formata_ptbr(v) for v in valores],
    textposition="top center",
    textfont=dict(size=11, color='#555'),
    hovertemplate='<b>Ano:</b> %{x}<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
))

# 3. Layout
fig.update_layout(
    title="Área de Energia Eólica no Brasil por Ano",
    xaxis_title="Ano",
    yaxis_title="Área (ha)",
    template="plotly_white",
    width=800,
    height=550,
    xaxis=dict(
        tickmode='linear',
        range=[min(anos) - 0.5, max(anos) + 0.5]
    ),
    yaxis=dict(
        range=[0, max(valores) * 1.2],
        tickformat=".2s",
        hoverformat=",.2f"
    ),
    margin=dict(t=80, b=60, l=60, r=40)
)

fig.show()

In [ ]:
# Célula 17